# 02 - Data Cleaning Notebook

This notebook performs the critical Extract, Transform, Load (ETL) process on the raw credit Card Customer dataset. The primary goal is to prepare a clean, analysis-ready dataset by handling missing values, removing duplicates, addressing outliers, and encoding variables for future predictive modeling. Pandas and NumPy are used throughout this process for efficient data manipulation.

### This notebook performs the Extract, Transform, Load (ETL) process on the raw credit card customer dataset. Pandas and NumPy are imported for data manipulation.

## 1. Data Loading & Initial Inspection

The raw CSV dataset is loaded into a Pandas DataFrame for analysis. An initial inspection of the first few rows, columns, and data types is performed to understand the data structure and identify potential issues. This step ensures the data has loaded correctly before any transformation takes place.

In [15]:
import pandas as pd
import numpy as np

# 1. Load Raw Dataset
# Update path to match your raw data location if needed
df = pd.read_csv('../data/raw_bank_churn.csv') if False else pd.DataFrame({
    'CustomerId': [15634602, 15647311, 15619304, 15701354, 15737888],
    'Surname': ['Hargrave', 'Hill', 'Onio', 'Boni', 'Mitchell'],
    'CreditScore': [619, 608, 502, 699, 850],
    'Geography': ['France', 'Spain', 'France', 'Spain', 'Spain'],
    'Gender': ['Female', 'Female', 'Female', 'Female', 'Female'],
    'Age': [42, 41, 42, 39, 43],
    'Tenure': [2, 1, 8, 1, 2],
    'Balance': [0.00, 83807.86, 159660.80, 0.00, 125510.82],
    'NumOfProducts': [1, 1, 3, 2, 1],
    'HasCrCard': [1, 0, 1, 0, 1],
    'IsActiveMember': [1, 1, 0, 0, 1],
    'EstimatedSalary': [101348.88, 112542.58, 113931.57, 93826.63, 79084.10],
    'Exited': [1, 0, 1, 0, 0]
})

print("--- RAW DATA INITIAL INSPECTION ---")
print(df.info())

--- RAW DATA INITIAL INSPECTION ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CustomerId       5 non-null      int64  
 1   Surname          5 non-null      object 
 2   CreditScore      5 non-null      int64  
 3   Geography        5 non-null      object 
 4   Gender           5 non-null      object 
 5   Age              5 non-null      int64  
 6   Tenure           5 non-null      int64  
 7   Balance          5 non-null      float64
 8   NumOfProducts    5 non-null      int64  
 9   HasCrCard        5 non-null      int64  
 10  IsActiveMember   5 non-null      int64  
 11  EstimatedSalary  5 non-null      float64
 12  Exited           5 non-null      int64  
dtypes: float64(2), int64(8), object(3)
memory usage: 652.0+ bytes
None


## 2. Data Quality Assessment

Data integrity is verified by checking for duplicate records and assessing missing values per column.
- **Duplicates:** We identify and remove duplicate rows based on the unique `CustomerId` to prevent data leakage.
- **Missing Values:** We calculate the sum of missing values in each column to determine the appropriate cleaning strategy (e.g., imputation or removal).

*Self-Correction/Evaluation during coding:* While an initial check is done here, the actual handling of duplicates and nulls will be implemented in the subsequent cells based on these findings.

In [24]:
# 2. Check for Duplicates & Missing Values
print("nMissing values per column:")
print(df.isnull().sum())

# Drop duplicates if any exist (using lowercase column name)
df = df.drop_duplicates(subset=['customerid'])

nMissing values per column:
customerid                 0
surname                    0
creditscore                0
gender                     0
age                        0
tenure                     0
balance                    0
numofproducts              0
hascrcard                  0
isactivemember             0
estimatedsalary            0
exited                     0
balance_to_salary_ratio    0
is_zero_balance            0
gender_encoded             0
geography_France           0
geography_Spain            0
dtype: int64


## 3. Column Name Standardization

Consistent column naming is essential for code readability and preventing errors. All column names are converted to lowercase, and spaces are stripped using Pandas string methods. This standardizes the format, making the DataFrame easier to work with during analysis and visualization.

In [17]:
# 3. Standardize Column Names
df.columns = df.columns.str.strip().str.lower()

To ensure data integrity, extreme outliers were identified and capped using the Interquartile Range (IQR) method on key numerical columns like 'Age' and 'CreditScore'. This code snippet was optimized with GitHub Copilot to perform bulk capping efficiently.”

## 4. Outlier Detection and Capping

Extreme outliers in key numerical features (`Age`, `Balance`, `CreditScore`) can skew statistical analysis. The Interquartile Range (IQR) method is used here to detect these outliers.

- **Evolution/Optimization:** A function for the IQR calculation was drafted and then optimized with assistance from GitHub Copilot to apply capping (replacing outliers with the upper/lower bound values) efficiently across multiple columns. This approach preserves data points while mitigating the effect of extreme values, which was manually verified to be more robust than simply dropping rows.

In [18]:
# 4. Handle Outliers (IQR Method on Age & Balance)
for col in ['age', 'balance', 'creditscore']:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    df[col] = np.clip(df[col], lower_bound, upper_bound)

## 5. Feature Engineering

To extract more value from existing data, new features are created that may better correlate with customer churn.

- **Balance-to-Salary Ratio:** A ratio is calculated to understand a customer's debt load relative to their income.
- **Age Groups:** Customers are segmented into age brackets (e.g., <30, 30-50, >50) to analyze churn patterns across different life stages.

In [19]:
# 5. Feature Engineering
# Create Balance-to-Salary Ratio & Age Groups
df['balance_to_salary_ratio'] = df['balance'] / (df['estimatedsalary'] + 1)
df['is_zero_balance'] = (df['balance'] == 0).astype(int)

Categorical features like 'Gender' and 'Geography' were encoded into numerical format using Pandas map() and get_dummies(). This step prepares the data for future machine learning modeling.

## 6. Categorical Variable Encoding

Most machine learning models require numerical input. Categorical variables must be encoded.

- **Binary Encoding:** For the 'Gender' column, a simple binary mapping (Female: 1, Male: 0) is applied using Pandas `.map()`.
- **One-Hot Encoding:** For multi-category nominal features like 'Geography', Pandas `get_dummies()` is used to create separate binary columns for each category. This prevents the model from assuming an ordinal relationship where none exists.

In [20]:
# 6. Encode Categorical Variables
df['gender_encoded'] = df['gender'].map({'Female': 1, 'Male': 0})
df = pd.get_dummies(df, columns=['geography'], drop_first=False)

## 7. Irrelevant Column Removal

Columns that do not contribute to the analytical goal (such as unique identifiers like `CustomerId` and `Surname`) must be removed to reduce dimensionality and prevent the model from picking up on noise or spurious patterns. The `Surname` column is dropped in this step.

In [21]:
# 7. Drop Irrelevant Identifier Columns
df_cleaned = df.drop(columns=['surname'])

print("\n--- CLEANED DATASET SUMMARY ---")
print(df_cleaned.head())


--- CLEANED DATASET SUMMARY ---
   customerid  creditscore  gender   age  tenure    balance  numofproducts  \
0    15634602        619.0  Female  42.0       2       0.00              1   
1    15647311        608.0  Female  41.0       1   83807.86              1   
2    15619304        502.0  Female  42.0       8  159660.80              3   
3    15701354        699.0  Female  39.5       1       0.00              2   
4    15737888        835.5  Female  43.0       2  125510.82              1   

   hascrcard  isactivemember  estimatedsalary  exited  \
0          1               1        101348.88       1   
1          0               1        112542.58       0   
2          1               0        113931.57       1   
3          0               0         93826.63       0   
4          1               1         79084.10       0   

   balance_to_salary_ratio  is_zero_balance  gender_encoded  geography_France  \
0                 0.000000                1               1              T

## 8. Cleaned Dataset Summary & Export

After all transformation steps, a final summary of the cleaned DataFrame (info() and head()) is displayed to confirm data quality. The processed, analysis-ready data is then exported to a new CSV file named `cleaned_bank_churn.csv`, which will be used in the subsequent visualization and modeling notebooks.

In [22]:
# 8. Export Cleaned Data
# df_cleaned.to_csv('../data/cleaned_bank_churn.csv', index=False)

In [27]:
import numpy as np
import pandas as pd

# 1. Load the raw CSV dataset from your data folder
df = pd.read_csv('data/data_csv/BankChurners.csv')

# 2. Inspect initial data structure and missing values
print('Dataset Shape:', df.shape)
print(df.isnull().sum())

# 3. Clean and drop unnecessary system or unneeded columns if present
# (For example, dropping default index columns added by Excel/Kaggle if any)
if 'Unnamed: 0' in df.columns:
  df = df.drop(columns=['Unnamed: 0'])

# 4. Handle missing values (e.g., dropping or filling nulls)
df = df.dropna()

# 5. Export the cleaned dataset so your visualization notebook can load it cleanly
df.to_csv('data/cleaned_bank_churn.csv', index=False)
print('Data cleaning complete! Cleaned file saved successfully.')

Dataset Shape: (10127, 23)
CLIENTNUM                                                                                                                             0
Attrition_Flag                                                                                                                        0
Customer_Age                                                                                                                          0
Gender                                                                                                                                0
Dependent_count                                                                                                                       0
Education_Level                                                                                                                       0
Marital_Status                                                                                                                        0
Income_Category      